# ESG RAG — 01: Indexing

This notebook turns the sustainability reports (PDFs) into a searchable **vector database**.

```
PDFs → extract text (page by page) → clean → [optional: transcribe charts] → chunks → embeddings → ChromaDB
```

It runs once, and again only when reports are added or the indexing settings change.

## Where this runs

The project was developed in **Google Colab** (free T4 GPU) and uses **Google Drive** as storage,
so nothing is lost when a Colab session ends. Everything lives in one Drive folder:

```
MyDrive/esg-rag/
├── data/
│   ├── pdfs/                 ← the reports
│   ├── pages_cache.json      ← extracted text, so each PDF is read only once
│   ├── vision_cache.json     ← chart and table transcriptions (optional vision step)
│   └── chunks*.json
├── chroma_db/                ← vector database (text only)
├── chroma_db_vision/         ← vector database (text + image transcriptions)
└── evals/                    ← questions.csv and the results of every evaluation run
```

**Colab and Drive are optional.** To run the project locally or on another platform:
1. Remove the `drive.mount(...)` line and set `PROJECT_DIR` to a local folder (e.g. `"./esg-rag"`).
2. Replace Colab Secrets (`userdata.get("ANTHROPIC_API_KEY")`) with an environment variable: `os.environ["ANTHROPIC_API_KEY"]`.
   In this notebook, the key is only needed for the optional vision step.
3. Install the libraries from `requirements.txt`. A GPU is recommended for the embeddings.

In Colab, the runtime should be set to a **GPU** (`Runtime → Change runtime type → T4`): embeddings are much faster on a GPU.

## Step 0 — The reports

The project uses the **2025** report of each company (the full report PDF, not the summary):

| Company | Source |
|---|---|
| Vale | https://www.vale.com/web/esg/sustainability-reports (Integrated Report) |
| Petrobras | https://sustentabilidade.petrobras.com.br (Relatório de Sustentabilidade) |
| Natura | https://ri.natura.com.br/sustentabilidade/publicacoes-socioambientais/ (Relatório Integrado) |

The PDFs are not included in the repository. The files must be named `company_year.pdf` (lowercase, no spaces):

```
natura_2025.pdf    petrobras_2025.pdf    vale_2025.pdf
```

The company and year are read from the file name and stored in the metadata of every chunk,
so more years (e.g. `vale_2024.pdf`) can be added without changing the code.

## Step 1 — Libraries

- `pymupdf4llm`: reads PDFs and keeps tables readable (as Markdown). Version 1.28.2 is pinned, because 2.0 on PyPI is an empty package.
- `sentence-transformers`: creates the embeddings
- `chromadb`: the vector database
- `anthropic`: the Claude API, used only by the optional image reader (Step 4c)

In [2]:
# pymupdf4llm 2.0 on PyPI is an empty package, so we pin the last working version
%pip install -q "pymupdf4llm==1.28.2" "pymupdf==1.28.2" chromadb sentence-transformers anthropic

## Step 2 — Settings, Google Drive and folders

Colab deletes everything when the session ends, so the project is kept in Google Drive.
The PDFs go in `MyDrive/esg-rag/data/pdfs/`.

**Experiments:** `CHUNK_SIZE` and `USE_VISION` can be changed here. Each setup is saved as a **separate database**
(for example `chroma_db_chunk3000` or `chroma_db_vision`), so the original database is never overwritten.
In `02_ask.ipynb` and `03_evals.ipynb`, the database is chosen with `DB_NAME`.

In [3]:
import os
from google.colab import drive

drive.mount("/content/drive")

# ── Chunking settings ── (change these to experiment; each size gets its own database)
CHUNK_SIZE = 1500      # maximum characters per chunk
CHUNK_OVERLAP = 200    # characters repeated between neighbor chunks

# ── Vision settings (Step 4c) ── charts and tables read by Claude from page images
USE_VISION = False             # False: skip the image reader, everything works like before
VISION_MODEL = "claude-haiku-5-5"
VISION_PAGES = "visual"        # "visual": only pages with charts, tables or large images | "all": every page
VISION_MAX_PAGES = None        # e.g. 5 for a cheap first test; None = no limit
VISION_ONLY_PAGES = None       # test specific pages only, e.g. {"petrobras_2025.pdf": [115, 143], "vale_2025.pdf": [17, 47]}

PROJECT_DIR = "/content/drive/MyDrive/esg-rag"
PDF_DIR = f"{PROJECT_DIR}/data/pdfs"
# Each setup gets its own database, so experiments never overwrite each other:
# chroma_db (default) · chroma_db_chunk3000 · chroma_db_vision · chroma_db_chunk3000_vision ...
SUFFIX = ("" if CHUNK_SIZE == 1500 else f"_chunk{CHUNK_SIZE}") + ("_vision" if USE_VISION else "")
DB_NAME = f"chroma_db{SUFFIX}"
CHUNKS_FILE = f"{PROJECT_DIR}/data/chunks{SUFFIX}.json"
DRIVE_DB_DIR = f"{PROJECT_DIR}/{DB_NAME}"   # final copy of the database (in Drive)
LOCAL_DB_DIR = "/content/chroma_db"         # we build the database here first (faster, safer)

os.makedirs(PDF_DIR, exist_ok=True)
print("Put your PDFs here:", PDF_DIR)
print(f"Chunk size: {CHUNK_SIZE} (overlap {CHUNK_OVERLAP}) | vision: {USE_VISION} → database: {DB_NAME}")

Mounted at /content/drive
Put your PDFs here: /content/drive/MyDrive/esg-rag/data/pdfs
Chunk size: 1500 (overlap 200) | vision: False → database: chroma_db


## Step 3 — Check the PDFs

Lists the PDFs and reads the company and year from each file name.

In [4]:
pdf_files = sorted(f for f in os.listdir(PDF_DIR) if f.lower().endswith(".pdf"))

documents = []
for filename in pdf_files:
    name = filename[:-4]                    # remove ".pdf"
    try:
        company, year = name.split("_")     # "vale_2024" -> "vale", "2024"
        documents.append({"file": filename, "company": company, "year": int(year)})
    except ValueError:
        print(f"⚠️ Skipping '{filename}': name it like company_year.pdf")

print(f"Found {len(documents)} reports:")
for doc in documents:
    print(f"  {doc['company']:<12} {doc['year']}   ({doc['file']})")

Found 3 reports:
  natura       2025   (natura_2025.pdf)
  petrobras    2025   (petrobras_2025.pdf)
  vale         2025   (vale_2025.pdf)


## Step 4 — Extract the text, page by page

Each page is extracted separately, so every piece of text keeps **the page it came from**.
That is what makes citations like "Petrobras 2025, page 87" possible.

⏱️ This is the slowest step: pages with images go through **OCR** (reading text from pictures), so large reports take several minutes.
The result is saved in `data/pages_cache.json`, so later runs load the pages from the cache in seconds.

⚠️ If a PDF is **replaced** by a different file with the same name, `pages_cache.json` must be deleted so it is extracted again.

In [5]:
import json
import pymupdf4llm

PAGES_CACHE = f"{PROJECT_DIR}/data/pages_cache.json"

def extract_pages(pdf_path):
    """Returns a list like [{"page": 1, "text": "..."}, {"page": 2, "text": "..."}]."""
    pages = pymupdf4llm.to_markdown(pdf_path, page_chunks=True)
    result = []
    for page_number, page in enumerate(pages, start=1):
        text = page["text"].strip()
        if len(text) > 50:                  # skip empty pages (covers, full-page photos)
            result.append({"page": page_number, "text": text})
    return result

# Load the cache (raw extracted pages) if it exists
cache = {}
if os.path.exists(PAGES_CACHE):
    with open(PAGES_CACHE, encoding="utf-8") as f:
        cache = json.load(f)

for doc in documents:
    if doc["file"] in cache:
        print(f"{doc['file']}: loaded from cache")
    else:
        print(f"Reading {doc['file']}...")
        cache[doc["file"]] = extract_pages(os.path.join(PDF_DIR, doc["file"]))
    # Work on a copy, so the cleaning step never changes the cache
    doc["pages"] = [dict(page) for page in cache[doc["file"]]]
    print(f"  {len(doc['pages'])} pages with text")

with open(PAGES_CACHE, "w", encoding="utf-8") as f:
    json.dump(cache, f, ensure_ascii=False)

natura_2025.pdf: loaded from cache
  89 pages with text
petrobras_2025.pdf: loaded from cache
  237 pages with text
vale_2025.pdf: loaded from cache
  115 pages with text


## Step 4b — Clean the text

The extracted text contains "noise" that hurts the search:

1. **OCR garbage:** text read from photos and icons, like `i<br>"|<br>——e`. Text from charts can be useful (numbers),
   so OCR text is kept only when it has enough real letters and digits.
2. **HTML tags:** `<br>`, `<mark>`, `<sup>`. For example, `m<sup>3</sup>` becomes `m3`.
3. **Repeated lines:** menus, headers and footers that appear on almost every page. The Natura report has a navigation menu
   ("Our reporting practices, A Message from Management…") on **every page**. Kept in, it makes every chunk look similar and confuses the search.

The cell prints the repeated lines it removed, as a check that nothing important was deleted.

In [6]:
import re
from collections import Counter

PICTURE_BLOCK = re.compile(r"<!-- Start of picture text -->(.*?)<!-- End of picture text -->", re.DOTALL)

def keep_useful_picture_text(match):
    content = match.group(1).replace("<br>", " ")
    useful_chars = sum(ch.isalnum() for ch in content)
    return content if useful_chars >= 30 else ""     # tiny OCR fragments are almost always garbage

def clean_text(text):
    text = PICTURE_BLOCK.sub(keep_useful_picture_text, text)
    text = re.sub(r"<sup>(.*?)</sup>", r"\1", text)   # m<sup>3</sup> -> m3
    text = re.sub(r"</?mark>", "", text)
    text = text.replace("<br>", " ")
    text = re.sub(r"[ \t]+\n", "\n", text)              # spaces at the end of lines
    text = re.sub(r"\n{3,}", "\n\n", text)              # too many empty lines
    return text.strip()

def remove_repeated_lines(pages, min_share=0.4):
    """Removes short lines that appear on many pages (menus, headers, footers)."""
    counts = Counter()
    for page in pages:
        counts.update({line.strip() for line in page["text"].splitlines() if line.strip()})
    limit = max(3, int(len(pages) * min_share))
    repeated = {
        line for line, n in counts.items()
        if n >= limit and len(line) < 100 and not line.startswith("|")   # never touch table rows
    }
    for page in pages:
        page["text"] = "\n".join(
            line for line in page["text"].splitlines() if line.strip() not in repeated
        )
    return repeated

for doc in documents:
    before = sum(len(p["text"]) for p in doc["pages"])
    for page in doc["pages"]:
        page["text"] = clean_text(page["text"])
    repeated = remove_repeated_lines(doc["pages"])
    for page in doc["pages"]:
        page["text"] = re.sub(r"\n{3,}", "\n\n", page["text"]).strip()
    doc["pages"] = [p for p in doc["pages"] if len(p["text"]) > 50]   # drop pages that became empty
    after = sum(len(p["text"]) for p in doc["pages"])

    print(f"{doc['file']}: {before:,} → {after:,} characters ({100 * (before - after) / before:.0f}% removed), "
          f"{len(doc['pages'])} pages")
    print(f"  Repeated lines removed: {sorted(repeated)[:15]}{' ...' if len(repeated) > 15 else ''}\n")

natura_2025.pdf: 231,837 → 218,355 characters (6% removed), 88 pages
  Repeated lines removed: ['**10**', '**11**', '**14**', '**15**', '**6**', '**7**', '**8**', '**9**', '**Natura** Integrated Report **2025**']

petrobras_2025.pdf: 925,379 → 906,497 characters (2% removed), 236 pages
  Repeated lines removed: []

vale_2025.pdf: 417,098 → 393,118 characters (6% removed), 114 pages
  Repeated lines removed: ['**Performance**', '2025 **Annual Report**', 'Additional information', 'Our risk management', 'Overview']



One page of **cleaned** text, as an example. Tables appear in Markdown format (with `|` separators).

In [7]:
example = documents[0]["pages"][10]
print(f"{documents[0]['file']} — page {example['page']}\n")
print(example["text"][:1500])

natura_2025.pdf — page 12

3

**13**

**1** Our reporting practices

**2**

A Message from Management

**About us**

**4**

2025

Highlights

**5**

Governance, risks and ethics

Financial performance

Natura

Brand and portfolio strategy

R&D and Omnichannel and innovation customer centricity

New avenues for growth

**12**

Operations Our and logistics people

2030 Commitments

References

###### **Our brands**

GRI 2-6

Founded in 1886 and incorporated into Natura in 2020, Avon pioneered the global direct selling model in the beauty sector, creating opportunities for independence for millions of women. Its portfolio includes iconic lines in Makeup, a category in which it is the number one brand in Brazil, as well as Hair, Fragrance and Home & Style. A pioneer in supporting women’s causes, such as combating gender-based violence and breast cancer, the brand also stands out internationally for innovation, holding more than 1,000 patents. In 2026, Avon brought together its social vocat

## Step 4c — 👁️ Read charts and tables from page images (optional: `USE_VISION`)

Text extraction can't read **charts** (the numbers come out scrambled or without their labels) and struggles with complex **tables**.
When `USE_VISION = True`, this step:

1. **Finds the visual pages**: pages with tables, vector charts (many drawings) or large images.
2. **Renders each one as an image** and asks Claude Haiku to **transcribe** its charts and tables: titles, units and every value.
   Pages without data are answered with `NO_VISUAL_DATA` and ignored.
3. **Adds the transcription to the page text**, marked as `[Visual content transcribed from the page image]`.
   From then on it is normal text: it is chunked, embedded and searched like everything else, so it works for **Qwen and Claude**.

💲 **Cost:** roughly US$ 0.0005–0.001 per page (estimated US$ 0.20–0.45 for the three reports), **paid only once**:
every transcription is saved in `data/vision_cache.json`, so a page is never sent twice.
For a quick test, `VISION_ONLY_PAGES` (Step 2) limits the step to chosen pages, e.g.
`{"petrobras_2025.pdf": [115, 143], "vale_2025.pdf": [17, 47]}`, so the transcriptions can be checked against the PDF.

When `USE_VISION = False`, this cell does nothing.

In [8]:
if not USE_VISION:
    print("Vision step skipped (USE_VISION = False). Everything continues like before.")
else:
    import base64, json
    import pymupdf
    import anthropic
    from concurrent.futures import ThreadPoolExecutor, as_completed
    from google.colab import userdata

    client_claude = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
    VISION_CACHE = f"{PROJECT_DIR}/data/vision_cache.json"
    vision_cache = json.load(open(VISION_CACHE, encoding="utf-8")) if os.path.exists(VISION_CACHE) else {}

    VISION_PROMPT = '''This image is page {page} of {company}'s {year} sustainability report.
Transcribe ONLY the data shown in charts, graphs, infographics and tables on this page:
- For each chart: its title, units, and every label with its value (for example, each year and its number).
- For each table: a Markdown table with the same headers, rows and values.
- Keep the original language and number format, and copy numbers exactly. Do not round, interpret or add anything.
- Ignore photos, logos, navigation menus, page numbers and decorative elements.
If the page has no charts or tables with data, reply exactly: NO_VISUAL_DATA'''

    def is_visual_page(page):
        '''A page is "visual" if it has a table, many vector drawings (charts), or large images.'''
        if VISION_PAGES == "all":
            return True
        try:
            if page.find_tables().tables:
                return True
        except Exception:
            pass
        if len(page.get_drawings()) >= 30:              # vector charts are made of many shapes
            return True
        page_area = page.rect.width * page.rect.height
        image_area = sum(pymupdf.Rect(img["bbox"]).get_area() for img in page.get_image_info())
        return image_area / page_area >= 0.10            # images covering at least 10% of the page

    def transcribe(doc, page_number, png_bytes):
        response = client_claude.messages.create(
            model=VISION_MODEL,
            max_tokens=4096,
            messages=[{"role": "user", "content": [
                {"type": "image", "source": {"type": "base64", "media_type": "image/png",
                                             "data": base64.b64encode(png_bytes).decode()}},
                {"type": "text", "text": VISION_PROMPT.format(page=page_number, company=doc["company"].capitalize(),
                                                              year=doc["year"])},
            ]}],
        )
        text = "".join(b.text for b in response.content if b.type == "text").strip()
        u = response.usage
        return text, (u.input_tokens * 0.10 + u.output_tokens * 0.50) / 1_000_000   # Haiku 5.5 prices (USD)

    # 1. Find the visual pages that aren't in the cache yet, and render them as images
    jobs = []
    for doc in documents:
        pdf = pymupdf.open(os.path.join(PDF_DIR, doc["file"]))
        visual = 0
        for page in pdf:
            n = page.number + 1
            if VISION_ONLY_PAGES is not None and n not in VISION_ONLY_PAGES.get(doc["file"], []):
                continue                                     # test mode: only the chosen pages
            if VISION_ONLY_PAGES is None and not is_visual_page(page):
                continue
            visual += 1
            key = f"{doc['file']}:{n}"
            if key in vision_cache:
                continue
            zoom = 2000 / max(page.rect.width, page.rect.height)     # long edge ≈ 2,000 px: readable labels
            jobs.append((doc, n, key, page.get_pixmap(matrix=pymupdf.Matrix(zoom, zoom)).tobytes("png")))
        print(f"{doc['file']}: {visual} visual pages")
    if VISION_MAX_PAGES is not None:
        jobs = jobs[:VISION_MAX_PAGES]
    print(f"Pages to transcribe now: {len(jobs)} (already in the cache: {len(vision_cache)})")

    # 2. Transcribe them (4 at a time), saving the cache after every page
    total_cost = 0.0
    with ThreadPoolExecutor(max_workers=4) as pool:
        futures = {pool.submit(transcribe, doc, n, png): (doc, n, key) for doc, n, key, png in jobs}
        for i, future in enumerate(as_completed(futures), start=1):
            doc, n, key = futures[future]
            try:
                text, cost = future.result()
            except Exception as e:
                print(f"⚠️ {key}: {e} (not cached, it will be retried next time)")
                continue
            total_cost += cost
            vision_cache[key] = text
            with open(VISION_CACHE, "w", encoding="utf-8") as f:
                json.dump(vision_cache, f, ensure_ascii=False)
            if i % 10 == 0 or i == len(jobs):
                print(f"  {i}/{len(jobs)} pages done · cost so far ≈ US$ {total_cost:.3f}")

    # 3. Add the transcriptions to the page texts (pages with no text get created)
    added = 0
    for doc in documents:
        pages_by_number = {p["page"]: p for p in doc["pages"]}
        for key, text in vision_cache.items():
            file, n = key.rsplit(":", 1)
            if file != doc["file"] or not text or text.startswith("NO_VISUAL_DATA"):
                continue
            block = "\n\n[Visual content transcribed from the page image]\n" + text
            n = int(n)
            if n in pages_by_number:
                if "[Visual content transcribed from the page image]" in pages_by_number[n]["text"]:
                    continue                                   # already added (this cell was run before)
                pages_by_number[n]["text"] += block
            else:
                pages_by_number[n] = {"page": n, "text": block.strip()}
            added += 1
        doc["pages"] = sorted(pages_by_number.values(), key=lambda p: p["page"])
    print(f"\n✅ Visual content added to {added} pages · this run cost ≈ US$ {total_cost:.3f}")

Vision step skipped (USE_VISION = False). Everything continues like before.


## Step 5 — Split pages into chunks

A full page can be long and cover several topics, so it is split into smaller **chunks**:

- `chunk_size`: maximum characters per chunk (1,500)
- `overlap`: characters repeated between neighbor chunks (200), so a sentence cut at the border is not lost

Cuts happen only at a line break or a space, so chunks **never start or end in the middle of a word**,
and cutting at line breaks keeps table rows intact.

In [9]:
def split_into_chunks(text, chunk_size=1500, overlap=200):
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))

        # END of the chunk: prefer a line break, then a space (never cut a word in half)
        if end < len(text):
            last_newline = text.rfind("\n", start, end)
            last_space = text.rfind(" ", start, end)
            if last_newline > start + chunk_size // 2:
                end = last_newline
            elif last_space > start + chunk_size // 2:
                end = last_space

        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= len(text):
            break

        # START of the next chunk: go back `overlap` characters,
        # then move forward to the next line break or space, so it starts at a whole word
        next_start = end - overlap
        newline = text.find("\n", next_start, end)
        space = text.find(" ", next_start, end)
        if newline != -1:
            next_start = newline + 1
        elif space != -1:
            next_start = space + 1
        else:
            next_start = end            # no clean place found: start right after this chunk
        start = next_start
    return chunks

all_chunks = []
for doc in documents:
    for page in doc["pages"]:
        for i, chunk_text in enumerate(split_into_chunks(page["text"], chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)):
            all_chunks.append({
                "id": f"{doc['company']}_{doc['year']}_p{page['page']}_c{i}",
                "text": chunk_text,
                "metadata": {
                    "company": doc["company"],
                    "year": doc["year"],
                    "page": page["page"],
                    "source": doc["file"],
                },
            })

print(f"Total chunks: {len(all_chunks)}")
print("\nExample chunk:")
print(all_chunks[0]["id"], all_chunks[0]["metadata"])
print(all_chunks[0]["text"][:500])

Total chunks: 1369

Example chunk:
natura_2025_p2_c0 {'company': 'natura', 'year': 2025, 'page': 2, 'source': 'natura_2025.pdf'}
###### **Contents**

Integrated Report **2025**

8 9 10 11 12 13 14

|Our reporting practices|**_3_**|
|---|---|
|A Message from Management|**_7_**|
|About us|**_11_**|
|2025 Highlights|**_16_**|
|Governance, risks and ethics|**_21_**|
|Financial performance|**_25_**|
|Natura: leader in beauty and personal care in Latin America|**_29_**|
|Brand and portfolio strategy|**_33_**|
|Research, development and innovation|**_39_**|
|Omnichannel and customer centricity|**_45_**|
|New avenues for growth|*


The chunks are also saved to Drive as JSON, so they can be reused (e.g. for keyword search) without extracting the PDFs again.

In [10]:
import json

with open(CHUNKS_FILE, "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, ensure_ascii=False)

print("Saved", len(all_chunks), "chunks to", CHUNKS_FILE)

Saved 1369 chunks to /content/drive/MyDrive/esg-rag/data/chunks.json


## Step 6 — Create the embeddings

An **embedding** is a list of numbers that represents the *meaning* of a text.
Texts with similar meanings get similar numbers, even in different languages.

The model is `BAAI/bge-m3`, an open multilingual model that works well in Portuguese and English
(the reports are in both, and so are the questions). The first run downloads it (about 2 GB).
The cell also checks that no chunk is longer than the model's token limit, since longer text would be silently cut.

In [11]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    print("⚠️ No GPU found. It will work, but slowly. Runtime → Change runtime type → T4 GPU")

model = SentenceTransformer("BAAI/bge-m3", device=device)
# bge-m3 reads at most max_seq_length tokens per chunk and IGNORES the rest.
# ~1,500 characters ≈ 400-500 tokens, so 1024 is enough; bigger chunks need a bigger limit.
model.max_seq_length = 1024 if CHUNK_SIZE <= 2500 else 2048
BATCH = 16 if CHUNK_SIZE <= 2500 else 8   # longer texts use more GPU memory

# Check: are any chunks longer than the limit? (they would be cut off)
token_counts = [len(model.tokenizer(c["text"])["input_ids"]) for c in all_chunks]
too_long = sum(n > model.max_seq_length for n in token_counts)
print(f"Tokens per chunk: average {sum(token_counts) / len(token_counts):.0f}, max {max(token_counts)}"
      f" | chunks over the {model.max_seq_length}-token limit: {too_long}")

texts = [chunk["text"] for chunk in all_chunks]
embeddings = model.encode(
    texts,
    batch_size=BATCH,
    show_progress_bar=True,
    normalize_embeddings=True,   # needed for cosine similarity
)

print("Embeddings shape:", embeddings.shape)   # (number of chunks, 1024)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.27GB            

... (10 outputs left out)

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Tokens per chunk: average 283, max 688 | chunks over the 1024-token limit: 0


Batches:   0%|          | 0/86 [00:00<?, ?it/s]

Embeddings shape: (1369, 1024)


## Step 7 — Save everything in ChromaDB

ChromaDB stores each chunk with its text, embedding and metadata (company, year, page), using cosine similarity.

The database is built on Colab's local disk first and then copied to Drive,
because databases don't always work well when written directly to Google Drive.

In [12]:
import shutil
import chromadb

from chromadb.api.client import SharedSystemClient
SharedSystemClient.clear_system_cache()

# Start fresh every time this notebook runs
shutil.rmtree(LOCAL_DB_DIR, ignore_errors=True)

client = chromadb.PersistentClient(path=LOCAL_DB_DIR)
collection = client.create_collection(
    name="esg_reports",
    metadata={"hnsw:space": "cosine"},
)

# Add in batches (ChromaDB has a limit on how much you can add at once)
BATCH_SIZE = 1000
for start in range(0, len(all_chunks), BATCH_SIZE):
    batch = all_chunks[start:start + BATCH_SIZE]
    collection.add(
        ids=[c["id"] for c in batch],
        documents=[c["text"] for c in batch],
        metadatas=[c["metadata"] for c in batch],
        embeddings=embeddings[start:start + BATCH_SIZE].tolist(),
    )

print("Chunks in the database:", collection.count())

# Copy the finished database to Drive
shutil.rmtree(DRIVE_DB_DIR, ignore_errors=True)
shutil.copytree(LOCAL_DB_DIR, DRIVE_DB_DIR)
print("Database saved to Drive:", DRIVE_DB_DIR)

Chunks in the database: 1369
Database saved to Drive: /content/drive/MyDrive/esg-rag/chroma_db


## Step 8 — Test the search

No LLM yet: this only checks that the search finds relevant chunks.
A **distance** closer to 0 means more similar. The examples ask in Portuguese (Petrobras report, in Portuguese)
and in English (Vale report, in English); since bge-m3 is multilingual, questions also work across languages.

In [13]:
def search(question, n_results=3, company=None):
    query_embedding = model.encode([question], normalize_embeddings=True).tolist()
    where = {"company": company} if company else None   # optional filter
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results,
        where=where,
    )
    for text, meta, dist in zip(results["documents"][0],
                                results["metadatas"][0],
                                results["distances"][0]):
        print(f"📄 {meta['company']} {meta['year']}, page {meta['page']}  (distance: {dist:.3f})")
        print(text[:400].replace("\n", " "), "...\n")

search("Quais foram as emissões de gases de efeito estufa da Petrobras em 2025?")

📄 petrobras 2025, page 115  (distance: 0.356)
Em 2025, neutralizamos 183 mil tCO2, equivalente a 3,94 milhões MWh de energia elétrica renovável adquirida. No exterior, nossas emissões de escopo 2 totalizaram 132 tCO2, representando apenas 0,0003% de nossas emissões absolutas operacionais em 2025.  Realizamos também o monitoramento das emissões operacionais provenientes exclusivamente das nossas atividades de óleo e gás, desconsiderando as emi ...

📄 petrobras 2025, page 117  (distance: 0.357)
> 8  Os valores apresentados referem-se às principais emissões da nossa cadeia de valor e não consideram a utilização de créditos de carbono para a compensação das emissões de GEE da gasolina Petrobras Podium Carbono Neutro, calculadas com base em Avaliação do Ciclo de Vida, que totalizaram 297,9 mil tCO2e em 2025. 9  Embora o presente documento apresente somente as categorias 10 e 11 do escopo 3, ...

📄 petrobras 2025, page 220  (distance: 0.394)
no Relatorio de Sustentabilidade 2025 e Painel ASG

In [14]:
# A question in English about Portuguese documents (or the other way around)
search("How much water does Vale use?", company="vale")

📄 vale 2025, page 47  (distance: 0.361)
###### **Water performance**  In 2025, the Company withdrew 458.3 Mm³ (million cubic meters) of freshwater, representing a 12% increase in total withdrawals compared to 2024. Despite this, we also increased the volume of water withdrawn and returned unused to the environment or allocated to third parties (340.3 Mm³) by 12%, helping reduce pressure on water availability in the watershed.  Recircula ...

📄 vale 2025, page 47  (distance: 0.364)
**2025 water balance** In million m3 GRI 303-1, 303-3, 303-4, 303-5  Total withdrawal 458.3 Operational water water withdrawn 118.0 forpurposes productive 611.2 seueinater 40.5 [Bl Effluent discharge after treatment Non-operational water JOG. worerunued and retumeddirecty ° to the environment MI escurceproductionUse, beingwithdrawnreturnedprocess anddirectlythathasdid notonototherthe enterenvironm ...

📄 vale 2025, page 46  (distance: 0.400)
##### **3.2.1.3 Water**  GRI 3-3, 303-1, 303- 2, 303-3, 303-4, 303-5

### Check the image reader

How many pages were transcribed, how many chunks contain vision text, and what one chart looks like after transcription
(Petrobras, page 115, which can be compared with the chart in the PDF).

In [15]:
if not USE_VISION:
    print("USE_VISION = False: nothing to check.")
else:
    # 1) How many pages did Claude transcribe?
    print("Pages in vision cache:", len(vision_cache))
    print("With visual data:", sum(1 for t in vision_cache.values() if "NO_VISUAL_DATA" not in t))

    # 2) Did the transcriptions make it into the chunks?
    vision_chunks = [c for c in all_chunks if "[Visual content transcribed" in c["text"]]
    print("Chunks with vision text:", len(vision_chunks), "of", len(all_chunks))

    # 3) Is the database the vision one?
    print("DB_NAME:", DB_NAME, "| chunks in collection:", collection.count())

    # 4) Show the chart transcription for Petrobras page 115
    for c in all_chunks:
        if c["metadata"]["source"] == "petrobras_2025.pdf" and c["metadata"]["page"] == 115 \
           and "[Visual content transcribed" in c["text"]:
            print("\n=====", c["id"])
            print(c["text"][c["text"].find("[Visual content"):][:1500])

USE_VISION = False: nothing to check.


## Summary

- Text extracted from every report, page by page, and cleaned of OCR noise, HTML tags and repeated menus
- Optionally (`USE_VISION`), charts and tables transcribed from the page images
- Chunks with metadata (company, year, page) saved in `data/chunks*.json`
- A vector database (`chroma_db` or `chroma_db_vision`) that finds relevant passages by meaning

**Index size:** 3 reports, 438 pages of text, 1,369 chunks (text only) or 1,611 chunks (with vision: 406 pages sent to the image reader, 164 with charts or tables).

**Next:** `02_ask.ipynb` loads this database, retrieves the most relevant passages for a question, and sends them to Qwen or Claude to write an answer with citations.